# Huấn luyện YOLOv8n · Signa v1.1

Mô hình tạm để tích hợp backend/app và ưu tiên rà nhãn; đã chạy đủ 100 epoch trên L4.
Hồ sơ: `ai/experiments/yolov8n-640-v1.1-temp-s0/`; chưa phải so sánh chính thức, chưa TFLite.
Dùng COCO pretrained `yolov8n.pt`, 46 lớp danh mục v1, ảnh 640 px,
100 epoch (không vượt 200), patience 30, seed 0. Chọn `best.pt` chỉ theo
mAP@0.5:0.95 trên **val**. Không dùng test để chỉnh mô hình/ngưỡng.

1. Trong Colab chọn **Runtime → Change runtime type → T4 GPU hoặc L4 GPU**.
2. Tải `signa-v1.1.zip` vào thư mục gốc **My Drive**; giữ ít nhất 3 GB trống trên Drive.
3. Chạy lần lượt các ô. Chạy mới dùng ô 5, sau mất phiên dùng ô 6; không chạy cả hai.
4. Xem kết quả val, chốt cấu hình rồi mới bật xuất test ở ô cuối.

Dữ liệu giải nén trên SSD `/content/v1.1`; chỉ run và kết quả nằm trên Drive.
200 điểm là ngân sách **cả dự án**, không phải số epoch; Colab tính điểm tùy GPU/thời gian.
Theo dõi Compute units trong Colab, dừng sớm nếu cần; không có bảo đảm 100 epoch vừa số điểm còn lại.

Nguồn: [Thiết kế AI](https://github.com/hana200804/DA/blob/ef46f19/docs/04-thiet-ke-ai.md),
[dữ liệu v1.1](https://github.com/hana200804/DA/blob/ef46f19/ai/data/README.md),
[train/resume](https://docs.ultralytics.com/modes/train/),
[mã fitness của bản ghim](https://github.com/ultralytics/ultralytics/blob/v8.3.203/ultralytics/utils/metrics.py).


## 1. Cài đặt và lấy hợp đồng Signa

Giữ Torch/CUDA sẵn có của Colab; ghim `ultralytics==8.3.203` để tránh thay đổi
fitness/API giữa các lượt. Lấy helper `LabelMap`, đọc YOLO và xuất COCO từ commit dữ liệu đã merge;
không phụ thuộc notebook này đã push. Repo phải đọc được từ Colab.
Nếu Colab yêu cầu restart sau cài đặt, restart rồi chạy lại từ ô này.


In [ ]:
%pip install -q ultralytics==8.3.203 pyyaml

import subprocess
import sys
from pathlib import Path

REPO_COMMIT = "ef46f19f1a6924486426dfa180e88fcea022267a"
REPO = Path("/content/signa-source")
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "https://github.com/hana200804/DA.git", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", REPO_COMMIT], check=True)
sys.path.insert(0, str(REPO / "ai/src"))

## 2. Mount Drive, giải nén và kiểm tra dữ liệu

Chỉ giải nén ZIP Signa do nhóm tạo. Không đổi `names`, không chia lại tập.
Ô này kiểm tra manifest, số ảnh/box, nhãn YOLO và danh mục COCO trước khi tốn GPU.
Chạy lại sẽ đọc bản giải nén sẵn; khi thay ZIP phải xóa `/content/v1.1` hoặc đổi phiên runtime.


In [ ]:
import csv
import hashlib
import json
import math
import shutil
from zipfile import ZipFile

import torch
import ultralytics
import yaml
from google.colab import drive
from ultralytics import YOLO, settings
from ultralytics.data.utils import check_det_dataset
from ultralytics.utils.metrics import Metric

from signa_ai.dataset import read_yolo, relative_path
from signa_ai.detection import EVAL_CONFIDENCE, Box, Detection, to_coco_predictions
from signa_ai.labels import load_label_map

assert ultralytics.__version__ == "8.3.203", "Restart runtime rồi cài lại bản ghim."
assert torch.cuda.is_available(), "Hãy chọn T4 GPU trước khi chạy tiếp."
print("GPU:", torch.cuda.get_device_name(0))
settings.update({"wandb": False, "mlflow": False, "comet": False, "clearml": False})

drive.mount("/content/drive")
ARCHIVE = Path("/content/drive/MyDrive/signa-v1.1.zip")
DATASET = Path("/content/v1.1")
DRIVE_ROOT = Path("/content/drive/MyDrive/Signa/experiments")
assert ARCHIVE.is_file(), f"Thiếu {ARCHIVE}"
if not DATASET.exists():
    with ZipFile(ARCHIVE) as archive:
        for member in archive.namelist():
            path = relative_path(member)
            assert path.parts[0] == "v1.1", f"ZIP sai thư mục gốc: {member}"
        archive.extractall("/content")

label_map = load_label_map(REPO / "ai/data/labels.yaml")
assert label_map.version == 1 and len(label_map) == 46
NAMES = label_map.names
config_path = DATASET / "data.yaml"
config = yaml.safe_load(config_path.read_text())
assert config["names"] == NAMES, "data.yaml lệch danh mục v1."
config["path"] = str(DATASET)
config_path.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False))
check_det_dataset(str(config_path), autodownload=False)
assert (DATASET / "manifest.csv").read_bytes() == (REPO / "ai/data/manifests/v1.1.csv").read_bytes()

EXPECTED = {"train": (6679, 13283), "val": (1599, 2685), "test": (1700, 2664)}
GROUND_TRUTH = {}
for split, (image_count, box_count) in EXPECTED.items():
    assert config[split] == f"images/{split}"
    truth = json.loads((DATASET / f"coco/{split}.json").read_text())
    assert truth["info"]["label_map_version"] == label_map.version
    assert {c["id"]: c["name"] for c in truth["categories"]} == NAMES
    assert len(truth["images"]) == image_count and len(truth["annotations"]) == box_count
    assert len({im["id"] for im in truth["images"]}) == image_count
    assert len({im["file_name"] for im in truth["images"]}) == image_count
    boxes = 0
    for image in truth["images"]:
        rel = relative_path(image["file_name"])
        assert rel.parts[:2] == ("images", split)
        assert (DATASET / rel).is_file(), str(rel)
        label = DATASET / "labels" / split / Path(rel).with_suffix(".txt").name
        boxes += len(read_yolo(label, label_map))
    assert boxes == box_count
    GROUND_TRUTH[split] = truth
    print(split, image_count, "ảnh,", boxes, "box hợp lệ")

# Bản ghim phải chọn và dừng theo mAP50-95 thuần, không pha mAP50.
probe = Metric()
probe.p, probe.r = torch.ones(1).numpy(), torch.ones(1).numpy()
probe.all_ap = torch.tensor([[0.9] + [0.1] * 9]).numpy()
assert math.isclose(probe.fitness(), probe.map, rel_tol=1e-6)
assert not math.isclose(probe.fitness(), probe.map50, rel_tol=1e-6)

## 3. Cấu hình và lưu phiên chạy

Run ghi **trực tiếp vào Drive**: `weights/{best,last}.pt`, `results.csv`, `args.yaml`,
ảnh batch và đồ thị. Mỗi epoch đã hoàn tất có checkpoint; epoch đang chạy khi mất phiên sẽ phải chạy lại.
Drive phải luôn được mount. Không đổi tên run khi resume; dùng tên mới nếu train lại từ đầu.

Đây là baseline tích hợp/rà nhãn, chưa phải lượt so sánh chính thức ba mô hình.
Tắt lật hai chiều, đổi sắc độ, mosaic; giữ co giãn/dịch chuyển và xoay 10° theo bảng chung.
Chưa bổ sung bộ tăng cường thời tiết/nén JPEG chung cho ba mô hình.


In [ ]:
import time
from datetime import UTC, datetime

RUN_NAME = "yolov8n-640-v1.1-temp-s0"
RUN_DIR = DRIVE_ROOT / RUN_NAME
EPOCHS = 100
BATCH = 16  # Nếu T4 báo OOM, giảm xuống 8; không giảm imgsz trong lượt này.
assert 1 < EPOCHS <= 200
TRAIN_ARGS = dict(
    data=str(config_path),
    epochs=EPOCHS,
    imgsz=640,
    batch=BATCH,
    device=0,
    patience=30,
    seed=0,
    deterministic=True,
    workers=2,
    cache=False,
    project=str(DRIVE_ROOT),
    name=RUN_NAME,
    exist_ok=True,
    save=True,
    val=True,
    plots=True,
    amp=True,
    fliplr=0.0,
    flipud=0.0,
    hsv_h=0.0,
    mosaic=0.0,
    mixup=0.0,
    scale=0.5,
    translate=0.1,
    degrees=10.0,
)


def track_session(model):
    meta_path = RUN_DIR / "meta.json"
    meta = (
        json.loads(meta_path.read_text())
        if meta_path.exists()
        else {
            "repo_commit": REPO_COMMIT,
            "dataset_version": "v1.1",
            "label_map_version": label_map.version,
            "sessions": [],
        }
    )
    session = {
        "started_at": datetime.now(UTC).isoformat(),
        "gpu": torch.cuda.get_device_name(0),
        "torch": torch.__version__,
        "ultralytics": ultralytics.__version__,
        "gpu_hours": 0.0,
    }
    meta["sessions"].append(session)
    start = time.monotonic()

    def save_progress(trainer):
        session["gpu_hours"] = (time.monotonic() - start) / 3600
        session["epoch"] = trainer.epoch + 1
        meta_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2))
        if trainer.fitness == trainer.best_fitness:
            (RUN_DIR / "best-selection.json").write_text(
                json.dumps(
                    {
                        "epoch": trainer.epoch + 1,
                        "fitness": trainer.fitness,
                        "val": {
                            k: float(v)
                            for k, v in trainer.metrics.items()
                            if k.startswith("metrics/")
                        },
                    },
                    indent=2,
                )
            )
        trainer.plot_metrics()  # Có results.png trên Drive ngay cả khi phiên sau bị ngắt.

    model.add_callback("on_model_save", save_progress)
    model.add_callback("on_train_end", save_progress)

## 4. Chạy thử một epoch trước

Ô này dùng 5% train, vẫn kiểm tra trên val, chỉ lưu `/content/signa-smoke`.
Xem `train_batch0.jpg`: box phải nằm trên biển và mã phải đúng.
Sau khi đạt, chạy ô 5; smoke không tiêu thụ epoch của run chính, cũng không được dùng làm kết quả báo cáo.
Sau mất phiên, bỏ qua smoke và ô 5, chạy ô 6 để resume.


In [ ]:
from IPython.display import Image, display

smoke = YOLO("yolov8n.pt")
smoke_args = {
    **TRAIN_ARGS,
    "epochs": 1,
    "fraction": 0.05,
    "project": "/content/signa-smoke",
    "name": RUN_NAME,
}
smoke.train(**smoke_args)
display(Image(filename=str(Path(smoke.trainer.save_dir) / "train_batch0.jpg")))
del smoke
torch.cuda.empty_cache()

## 5. Huấn luyện mới · bỏ qua nếu resume

Không ghi đè run có sẵn. Trong lúc chạy có thể xem log/đồ thị trên Drive.
Chỉ `best.pt` dùng cho suy luận; `last.pt` dành cho tiếp tục phiên bị ngắt.


In [ ]:
assert not RUN_DIR.exists(), f"Run đã tồn tại: {RUN_DIR}. Dùng ô resume hoặc đổi RUN_NAME."
RUN_DIR.mkdir(parents=True)
(RUN_DIR / "config.yaml").write_text(yaml.safe_dump(TRAIN_ARGS, sort_keys=False))
subprocess.run(
    [sys.executable, "-m", "pip", "freeze"],
    check=True,
    stdout=(RUN_DIR / "requirements.txt").open("w"),
)
model = YOLO("yolov8n.pt")
track_session(model)
model.train(**TRAIN_ARGS)

## 6. Tiếp tục từ last.pt trên Drive · chỉ chạy sau mất phiên

Trong runtime mới, chạy lại ô 1–3 rồi chạy ô này; bỏ qua ô 4–5.
Khôi phục optimizer và epoch từ checkpoint với `resume=True`, hoàn thành **tổng số epoch ban đầu**.
Không dùng `best.pt` để resume. Lượt đã hoàn tất/early stop bị bỏ optimizer sẽ được chặn;
muốn fine-tune thêm phải tạo run mới từ `best.pt` và ghi cấu hình riêng.
`results.csv` trên Drive giữ các epoch cũ. Ultralytics khôi phục `best_fitness`;
bộ đếm patience của phiên trước không được lưu đầy đủ, nên có thể dừng muộn hơn khi resume.


In [ ]:
last_path = RUN_DIR / "weights/last.pt"
assert last_path.is_file(), f"Chưa có checkpoint: {last_path}"
# Chỉ nạp checkpoint do nhóm tạo; checkpoint PyTorch có thể thực thi mã khi nạp.
checkpoint = torch.load(last_path, map_location="cpu", weights_only=False)
assert checkpoint.get("epoch", -1) >= 0 and checkpoint.get("optimizer") is not None, (
    "Run đã hoàn tất hoặc checkpoint không thể resume. Dùng best.pt với run mới."
)
assert checkpoint["train_args"]["data"] == str(config_path)
assert Path(checkpoint["train_args"]["project"]) == DRIVE_ROOT
assert checkpoint["train_args"]["name"] == RUN_NAME
assert checkpoint["train_args"]["epochs"] <= 200
del checkpoint
model = YOLO(str(last_path))
track_session(model)
model.train(resume=True, device=0, batch=BATCH)

## 7. Gói mô hình tạm để bàn giao

Đọc điểm **val thực tế** từ callback chọn checkpoint, đối chiếu log `results.csv`;
không gán số test/thực địa chưa đo.
`handoff/` chứa `best.pt`, `labels.yaml`, `labels.json`, `model_card.json` và cấu hình.
Backend dùng `.pt`; thành viên app có thể tích hợp qua API backend.
Gói này chưa có TFLite và chưa xác minh FPS/kích thước trên điện thoại để chạy camera offline.
Giữ cả `meta.json`, `requirements.txt`, `results.csv` và đồ thị ở thư mục run để truy vết.


In [ ]:
from dataclasses import asdict

best_path = RUN_DIR / "weights/best.pt"
assert best_path.is_file(), "Cần ít nhất một epoch hoàn tất."
best_model = YOLO(str(best_path))
assert best_model.names == NAMES
selected = json.loads((RUN_DIR / "best-selection.json").read_text())
actual_args = yaml.safe_load((RUN_DIR / "args.yaml").read_text())
HANDOFF = RUN_DIR / "handoff"
HANDOFF.mkdir(exist_ok=True)
shutil.copy2(best_path, HANDOFF / "best.pt")
shutil.copy2(REPO / "ai/data/labels.yaml", HANDOFF / "labels.yaml")
shutil.copy2(RUN_DIR / "args.yaml", HANDOFF / "args.yaml")
(HANDOFF / "labels.json").write_text(
    json.dumps(
        {
            "version": label_map.version,
            "classes": [asdict(c) for c in label_map.classes],
        },
        ensure_ascii=False,
        indent=2,
    )
)
(HANDOFF / "model_card.json").write_text(
    json.dumps(
        {
            "modelName": "yolov8",
            "modelVersion": "yolov8n-v1.1-temp",
            "weightsSha256": hashlib.sha256((HANDOFF / "best.pt").read_bytes()).hexdigest(),
            "status": "temporary",
            "datasetVersion": "v1.1",
            "labelMapVersion": label_map.version,
            "repoCommit": REPO_COMMIT,
            "trainingConfig": actual_args,
            "selection": {
                "split": "val",
                "metric": "mAP@0.5:0.95",
                "epoch": int(selected["epoch"]),
            },
            "val": selected["val"],
            "test": None,
            "field": None,
            "deviceFps": None,
            "limitations": [
                "Mô hình tạm; chưa so sánh ba kiến trúc",
                "Chưa xuất/đo TFLite",
                "Test cùng nguồn VN82; lớp P.124c ít mẫu val/test",
            ],
        },
        ensure_ascii=False,
        indent=2,
    )
)
print("Gói bàn giao:", HANDOFF)
print("Epoch chọn:", selected["epoch"], "val mAP50-95:", selected["fitness"])
display(Image(filename=str(RUN_DIR / "results.png")))

## 8. Xuất box nghi sai nhãn ra CSV

Quét **train** bằng `best.pt`: ghép box nguồn–dự đoán theo IoU giảm dần, không xét lớp,
mỗi box chỉ ghép một lần. IoU ≥ 0,5; khác mã với confidence ≥ 0,8 → `different_class`;
không có box ghép ở confidence ≥ 0,25 → `not_detected` (mã dự đoán trống, confidence 0).
Đây là ngưỡng gợi ý rà nhãn, không phải ngưỡng vận hành.

`box_index` bắt đầu từ 1 trong file nhãn **v1.1 đã ánh xạ**, không phải chỉ số nhãn VN82 gốc.
Mã gốc là mã nguồn v1.1 đang kiểm tra. CSV ghi ảnh tương đối từ `/content/v1.1`.
Không tự sửa nhãn. Mô hình đã học train nên có thể nhớ nhãn sai hoặc bỏ sót biển nhỏ;
CSV chỉ ưu tiên ảnh cần xem, không chứng minh nhãn sai.
Vẫn phải lấy 10% ảnh để hai người gán nhãn độc lập theo docs/05, gồm cả ảnh không bị gắn cờ.
Không dùng test để sửa nhãn/chọn ngưỡng trong lượt này.


In [ ]:
def iou_xywh(a, b):
    left, top = max(a[0], b[0]), max(a[1], b[1])
    right, bottom = min(a[0] + a[2], b[0] + b[2]), min(a[1] + a[3], b[1] + b[3])
    intersection = max(0, right - left) * max(0, bottom - top)
    union = a[2] * a[3] + b[2] * b[3] - intersection
    return intersection / union if union > 0 else 0.0


def suspicious_boxes(source_boxes, predictions, names, match_iou=0.5, high_conf=0.8):
    pairs = sorted(
        (
            (iou_xywh(box["bbox"], pred["bbox"]), i, j)
            for i, box in enumerate(source_boxes)
            for j, pred in enumerate(predictions)
        ),
        reverse=True,
    )
    matched, used = {}, set()
    for overlap, i, j in pairs:
        if overlap < match_iou:
            break
        if i not in matched and j not in used:
            matched[i] = j
            used.add(j)
    rows = []
    for i, source in enumerate(source_boxes):
        pred = predictions[matched[i]] if i in matched else None
        if pred is None or (
            pred["category_id"] != source["category_id"] and pred["score"] >= high_conf
        ):
            rows.append(
                {
                    "box_index": i + 1,
                    "original_code": names[source["category_id"]],
                    "predicted_code": names[pred["category_id"]] if pred else "",
                    "confidence": pred["score"] if pred else 0.0,
                    "reason": "different_class" if pred else "not_detected",
                }
            )
    return rows


def result_predictions(result, image_id):
    detections = [
        Detection(Box.from_xyxy(*xyxy), NAMES[int(cls)], float(score))
        for xyxy, cls, score in zip(
            result.boxes.xyxy.cpu().tolist(),
            result.boxes.cls.cpu().tolist(),
            result.boxes.conf.cpu().tolist(),
            strict=True,
        )
    ]
    return to_coco_predictions(image_id, detections, label_map)


QC_CONF, QC_HIGH_CONF, QC_IOU = 0.25, 0.8, 0.5
qc_path = RUN_DIR / "suspected-labels-train.csv"
fields = ["image", "box_index", "original_code", "predicted_code", "confidence", "reason"]
with qc_path.open("w", encoding="utf-8-sig", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=fields)
    writer.writeheader()
    for image in GROUND_TRUTH["train"]["images"]:
        path = DATASET / image["file_name"]
        label = DATASET / "labels/train" / path.with_suffix(".txt").name
        source_boxes = [
            {"category_id": b.class_id, "bbox": b.coco(image["width"], image["height"])}
            for b in read_yolo(label, label_map)
        ]
        result = best_model.predict(
            str(path), imgsz=640, conf=QC_CONF, iou=0.7, max_det=300, device=0, verbose=False
        )[0]
        predictions = result_predictions(result, image["id"])
        for row in suspicious_boxes(source_boxes, predictions, NAMES, QC_IOU, QC_HIGH_CONF):
            writer.writerow({"image": image["file_name"], **row})
print("CSV rà tay:", qc_path)

## 9. Xuất COCO JSON cho nhánh đánh giá · ô cuối

Xuất val trước. Sau khi đã chốt cấu hình/ngưỡng trên val, đổi `EXPORT_TEST=True` và chạy lại ô này
để có cả `predictions-val.json` và `predictions-test.json`.
Không chỉnh mô hình theo điểm test. Giữ confidence 0,001, NMS IoU 0,7, tối đa 300 box/ảnh.

`image_id` lấy nguyên từ `coco/<split>.json`, **không lấy từ tên ảnh**;
`category_id` là ID 0–45 của `labels.yaml`, **không cộng 1**.
`bbox=[x_min,y_min,width,height]` theo pixel ảnh gốc. Ảnh không phát hiện không có dòng dự đoán;
nhãn COCO kèm theo vẫn giữ đủ ảnh để bộ đánh giá tính false negative.
Chuyển cả thư mục `evaluation/` cho nhánh ai/3-evaluation, dùng `pycocotools.COCO.loadRes`.
Nếu JSON là `[]`, bộ đánh giá cần xử lý trường hợp không có dự đoán trước khi gọi `loadRes`.


In [ ]:
EXPORT_TEST = False  # Chỉ bật sau khi chốt trên val; không chọn mô hình bằng test.
EVALUATION = RUN_DIR / "evaluation"
EVALUATION.mkdir(exist_ok=True)
for split in ("val", "test") if EXPORT_TEST else ("val",):
    truth = GROUND_TRUTH[split]
    predictions = []
    for image in truth["images"]:
        result = best_model.predict(
            str(DATASET / image["file_name"]),
            imgsz=640,
            conf=EVAL_CONFIDENCE,
            iou=0.7,
            max_det=300,
            device=0,
            verbose=False,
        )[0]
        assert result.orig_shape == (image["height"], image["width"])
        predictions.extend(result_predictions(result, image["id"]))
    output = EVALUATION / f"predictions-{split}.json"
    output.write_text(json.dumps(predictions, ensure_ascii=False, allow_nan=False))
    shutil.copy2(DATASET / f"coco/{split}.json", EVALUATION / f"ground-truth-{split}.json")
    print(split, len(predictions), "dự đoán →", output)
shutil.copy2(REPO / "ai/data/labels.yaml", EVALUATION / "labels.yaml")
(EVALUATION / "export-config.json").write_text(
    json.dumps(
        {
            "datasetVersion": "v1.1",
            "labelMapVersion": label_map.version,
            "weights_sha256": hashlib.sha256(best_path.read_bytes()).hexdigest(),
            "conf": EVAL_CONFIDENCE,
            "iou": 0.7,
            "max_det": 300,
            "imgsz": 640,
            "test_exported": EXPORT_TEST,
        },
        indent=2,
    )
)
print("Đã lưu trên Drive:", RUN_DIR)